# Work in progress: Batched LLM inference

Note: This notebook is a work in progress.

* TK - this notebook builds off of the LLM fine-tuning tutorial - https://www.learnhuggingface.com/notebooks/hugging_face_llm_full_fine_tune_tutorial 

In [1]:
import time

print(f"Last updated: {time.ctime()}")

Last updated: Wed Apr  8 01:18:47 2026


## TK - Intro/overview

TK - split this into another notebook

Right now our model only inferences on one sample at a time but as is the case with many machine learning models, we could perform inference on multiple samples (also referred to as a batch) to significantly improve throughout.

In batched inference mode, your model performs predictions on X number of samples at once, this can dramatically improve sample throughput.

The number of samples you can predict on at once will depend on a few factors: 

* The size of your model (e.g. if your model is quite large, it may only be able to predict on 1 sample at time)
* The size of your compute VRAM (e.g. if your compute VRAM is already saturated, add multiple samples at a time may result in errors)
* The size of your samples (if one of your samples is 100x the size of others, this may cause errors with batched inference)

To find an optimal batch size for our setup, we can run an experiment:

* Loop through different batch sizes and measure the throughput for each batch size.
    * Why do we do this?
        * It's hard to tell the ideal batch size ahead of time.
        * So we experiment from say 1, 2, 4, 8, 16, 32, 64 batch sizes and see which performs best.
        * Just because we may get a speed up from using batch size 8, doesn't mean 64 will be better. 

UPTOHERE:

* Next: write out the batching notebook from scratch and make sure it works
* Load dataset
* Load model
* 3 methods of batching
    * Manual batching with manual chunks
    * Auto batching with pipeline
    * Batching with progress tracking and KeyDataset
* Eval the batched samples to make sure they match the original (e.g. string matching for simplicity to make sure they are the same level)
* Compare performance with a plot

## TK - Load Dataset



In [2]:
from datasets import load_dataset

DATASET_ID = "mrdbourke/FoodExtract-1k"

print(f"[INFO] Loading dataset: {DATASET_ID}")
dataset = load_dataset(DATASET_ID)

print(f"[INFO] Number of samples in the dataset: {len(dataset['train'])}")

[INFO] Loading dataset: mrdbourke/FoodExtract-1k
[INFO] Number of samples in the dataset: 1420


In [3]:
import random
random_sample = random.choice(dataset['train'])
random_sample

{'sequence': 'animal the definitive visual guide',
 'image_url': 'https://www.libertybooks.com/image/catalog/82479.jpg',
 'class_label': 'not_food',
 'source': 'qwen2vl_open_dataset',
 'char_len': 34.0,
 'word_count': 5.0,
 'syn_or_real': 'real',
 'uuid': 'cdb83c67-fc97-479d-bd7c-279b26a00e02',
 'gpt-oss-120b-label': "{'is_food_or_drink': False, 'tags': [], 'food_items': [], 'drink_items': []}",
 'gpt-oss-120b-label-condensed': 'food_or_drink: 0\ntags: \nfoods: \ndrinks:',
 'target_food_names_to_use': None,
 'caption_detail_level': None,
 'num_foods': None,
 'target_image_point_of_view': None}

In [4]:
# Create helper function to turn samples into prompt and completion pairs
def sample_to_prompt_completion(sample):
    """Helper function to convert an input sample to prompt-completion style."""
    return {
        "prompt": [
            {"role": "user", "content": sample["sequence"]}, # load the sequence from the dataset
        ],
        "completion": [
            {"role": "assistant", "content": sample["gpt-oss-120b-label-condensed"]} # load the condensed label from the ground truth
        ]
    }

sample_to_prompt_completion(random_sample)

{'prompt': [{'role': 'user', 'content': 'animal the definitive visual guide'}],
 'completion': [{'role': 'assistant',
   'content': 'food_or_drink: 0\ntags: \nfoods: \ndrinks:'}]}

In [5]:
# Map the helper function to the dataset
dataset = dataset.map(sample_to_prompt_completion,
                      batched=False)

dataset["train"][42]

{'sequence': 'another optional quest takes place on windfall island during the night time play the song of passing a number of times and each time, glance towards the sky',
 'image_url': 'https://portforward.com/games/walkthroughs/The-Legend-of-Zelda-The-Wind-Waker/The-Legend-of-Zelda-The-Wind-Waker-large-430.jpg',
 'class_label': 'not_food',
 'source': 'qwen2vl_open_dataset',
 'char_len': 156.0,
 'word_count': 28.0,
 'syn_or_real': 'real',
 'uuid': 'bbac79ce-df1f-48b8-891c-752809be11c7',
 'gpt-oss-120b-label': "{'is_food_or_drink': 'false', 'tags': [], 'food_items': [], 'drink_items': []}",
 'gpt-oss-120b-label-condensed': 'food_or_drink: 0\ntags: \nfoods: \ndrinks:',
 'target_food_names_to_use': None,
 'caption_detail_level': None,
 'num_foods': None,
 'target_image_point_of_view': None,
 'prompt': [{'content': 'another optional quest takes place on windfall island during the night time play the song of passing a number of times and each time, glance towards the sky',
   'role': 'use

In [6]:
# Create a train/test split
dataset = dataset["train"].train_test_split(test_size=0.2, 
                                            shuffle=False,
                                            seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'prompt', 'completion'],
        num_rows: 1136
    })
    test: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'prompt', 'completion'],
        num_rows: 284
    })
})

## TK - Load the model and tokenizer

Our model is hosted here: https://huggingface.co/mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1"

print(f"[INFO] Loading tokenizer and model from: {MODEL_ID}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_ID)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

print(f"[INFO] Tokenizer and model loaded from: {MODEL_ID}")

# Check our model
model

[INFO] Loading tokenizer and model from: mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1


/home/mrdbourke/miniforge3/envs/ai/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


[INFO] Tokenizer and model loaded from: mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1


Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [8]:
# Turn our loaded model into a pipeline for easy handling of preprocessing
from transformers import pipeline

loaded_model_pipeline = pipeline(task="text-generation",
                                 model=model,
                                 tokenizer=tokenizer)

loaded_model_pipeline

Device set to use cuda:0


* TK - we're going to perform batched inference on the test dataset, let's first format it with the prompt/chat template.
* TK - show before and after of the sample with/without the prompt

In [12]:
# Format the test dataset with the prompt template
test_dataset = dataset["test"]

def format_input_prompt(sample):
    """Helper function to add the tokenizer chat template to the input prompt."""
    formatted_prompt = loaded_model_pipeline.tokenizer.apply_chat_template(sample["prompt"],
                                                                           tokenize=False,
                                                                           add_generation_prompt=True)
    return {"formatted_prompt": formatted_prompt}

test_dataset = test_dataset.map(format_input_prompt, batched=False)
test_dataset[42]

Map:   0%|          | 0/284 [00:00<?, ? examples/s]

{'sequence': "This image shows the back of a food package with detailed cooking instructions, ingredients, and nutrition information. The package is held in a person's hand, and the background includes a concrete floor and part of a shoe.\n\n**Cooking Instructions:**\n- Ingredients listed include 1 cup snow peas, 1/2 cup frozen edamame, 2 cloves garlic, 2cm piece ginger, 1/4 cup stock of choice, 180g udon noodles, 3 spring onions, and 2 tablespoons sesame seeds.\n- Instructions mention cooking snow peas and edamame, adding oil, garlic, ginger, and stock, and then adding cooked udon noodles and tossing with protein, sesame seeds, and spring onions.\n\n**Nutrition Information:**\n- Servings per pack: 4\n- Serving size: 44g\n- Energy: 88kJ (21kcal) per serve, 201kJ (48kcal) per 100g\n- Protein: 0.4g per serve, 0.9g per 100g\n- Fat, Total: 0.6g per serve, 1.4g per 100g\n- Saturated Fat: 0.1g per serve, 0.3g per 100g\n- Carbohydrate: 3.4g per serve, 7.8g per 100g\n- Sugars: 1.1g per serve, 

In [ ]:
# TK - Run the model on a random sample
random_test_sample = random.choice(test_dataset)

random_sample_input_prompt_formatted = loaded_model_pipeline.tokenizer.apply_chat_template(random_test_sample["prompt"],
                                                                                           tokenize=False,
                                                                                           add_generation_prompt=True)    

random_sample_generated_output = loaded_model_pipeline(random_sample_input_prompt_formatted,
                                                       max_new_tokens=256,
                                                       disable_compile=True,
                                                       return_full_text=False) # only return the generated text, not the full prompt + generated

random_sample_generated_output

[{'generated_text': 'food_or_drink: 0\ntags: \nfoods: \ndrinks:'}]

## TK - Run batched inference with manual batching

We do manual batching to allow customization to the samples within the inference steps.

TK - three approaches:
1. Manual = full control but potentially error prone
2. Pipeline = automated but requires the input to be materialized as a list
3. KeyDataset = works directly with the dataset (good for larger datasets)

In [ ]:
import time 
from tqdm.auto import tqdm

all_outputs_manual = {}

BATCH_SIZES_TO_TEST = [1, 4, 8, 16, 32, 64, 128] # customize these based on the size of your model and GPU memory
VERBOSE = False

for BATCH_SIZE in BATCH_SIZES_TO_TEST:
    print(f"\n[INFO] Running inference with batch size: {BATCH_SIZE}")
    start_time = time.time()
    
    batched_outputs_list = []
    for batch_num in tqdm(range(round(len(test_dataset) / BATCH_SIZE)), desc=f"Batch size {BATCH_SIZE}"):

        # Calculate the target index numbers of the dataset to select for the current batch
        # Add a check to ensure we don't go out of bound of the dataset length
        idxs_to_select = [i for i in list(range(BATCH_SIZE * batch_num, BATCH_SIZE * (batch_num + 1))) if i < len(test_dataset)]
        if VERBOSE:
            print(f"[INFO] Working on indexes: {idxs_to_select}")

        # Select indexes from the dataset and extract the formatted prompts for the current batch
        batched_inputs = test_dataset.select(idxs_to_select)
        batched_formatted_prompts = [item["formatted_prompt"] for item in batched_inputs]

        # Perform inference with pipeline on a list of input prompts and store the outputs
        batched_outputs = loaded_model_pipeline(batched_formatted_prompts,
                                                batch_size=BATCH_SIZE,
                                                max_new_tokens=256,
                                                disable_compile=True,
                                                return_full_text=False) # only return the generated text, not the full prompt + generated text
        
        # Add the outputs for the current batch to the list of inputs (so we can compare the generated outputs with the inputs later on if we want)
        batched_inputs_list = list(batched_inputs)
        for i in range(len(batched_outputs)):
            batched_inputs_list[i]["generated_text"] = batched_outputs[i][0]["generated_text"]

        # Extend the list of outputs
        batched_outputs_list.extend(batched_inputs_list)
    
    end_time = time.time()
    total_time = end_time - start_time
    avg_time_per_sample = total_time / len(test_dataset)

    # Append the outputs and total time for the current batch to our results dictionary
    all_outputs_manual[BATCH_SIZE] = {"batched_outputs_list": batched_outputs_list, 
                                      "total_time": total_time,
                                      "avg_time_per_sample": avg_time_per_sample}

    print(f"[INFO] Total inference time for batch size {BATCH_SIZE}: {total_time:.2f} seconds")
    print(f"[INFO] Average inference time per sample for batch size {BATCH_SIZE}: {avg_time_per_sample:.2f} seconds")
    print("="*80 + "\n\n")


[INFO] Running inference with batch size: 1


Batch size 1:   0%|          | 0/284 [00:00<?, ?it/s]

[INFO] Total inference time for batch size 1: 138.93 seconds
[INFO] Average inference time per sample for batch size 1: 0.49 seconds



[INFO] Running inference with batch size: 4


Batch size 4:   0%|          | 0/71 [00:00<?, ?it/s]

[INFO] Total inference time for batch size 4: 52.60 seconds
[INFO] Average inference time per sample for batch size 4: 0.19 seconds



[INFO] Running inference with batch size: 8


...

[INFO] Total inference time for batch size 32: 31.88 seconds
[INFO] Average inference time per sample for batch size 32: 0.11 seconds



[INFO] Running inference with batch size: 64


Batch size 64:   0%|          | 0/4 [00:00<?, ?it/s]

[INFO] Total inference time for batch size 64: 24.65 seconds
[INFO] Average inference time per sample for batch size 64: 0.09 seconds



[INFO] Running inference with batch size: 128


Batch size 128:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Total inference time for batch size 128: 39.90 seconds
[INFO] Average inference time per sample for batch size 128: 0.14 seconds




In [55]:
for key, value in all_outputs_manual.items():
    print(f"Batch size: {key} | Total inference time: {value['total_time']:.2f} seconds | Total samples: {len(value['batched_outputs_list'])} | Average time per sample: {value['avg_time_per_sample']:.2f} seconds")

Batch size: 1 | Total inference time: 138.93 seconds | Total samples: 284 | Average time per sample: 0.49 seconds
Batch size: 4 | Total inference time: 52.60 seconds | Total samples: 71 | Average time per sample: 0.19 seconds
Batch size: 8 | Total inference time: 34.88 seconds | Total samples: 36 | Average time per sample: 0.12 seconds
Batch size: 16 | Total inference time: 24.36 seconds | Total samples: 18 | Average time per sample: 0.09 seconds
Batch size: 32 | Total inference time: 31.88 seconds | Total samples: 9 | Average time per sample: 0.11 seconds
Batch size: 64 | Total inference time: 24.65 seconds | Total samples: 4 | Average time per sample: 0.09 seconds
Batch size: 128 | Total inference time: 39.90 seconds | Total samples: 2 | Average time per sample: 0.14 seconds


In [ ]:
# TK - Evaluate the generated outputs 

## TK - Run batched inference with automatic batching with pipeline

Automatic batching is the simplest option. But it requires our inputs to be materialized to a list (this can cause memory issues if our dataset is large).


In [ ]:
import time

all_outputs_pipeline = {}

BATCH_SIZES_TO_TEST = [1, 4, 8, 16, 32, 64, 128] # customize these based on the size of your model and GPU memory
VERBOSE = False

# Turn our input prompts into a list for the pipeline
test_input_prompts = list(test_dataset["formatted_prompt"])
print(f"[INFO] Total number of samples to run through the pipeline: {len(test_input_prompts)}")

for BATCH_SIZE in BATCH_SIZES_TO_TEST:
    print(f"[INFO] Running inference with batch size: {BATCH_SIZE}")
    start_time = time.time()

    pipeline_outputs = loaded_model_pipeline(
        text_inputs=test_input_prompts,
        batch_size=BATCH_SIZE,
        max_new_tokens=256,
        return_full_text=False
    )
    print(f"[INFO] Total pipeline outputs: {len(pipeline_outputs)}")

    end_time = time.time()
    total_time = end_time - start_time
    avg_time_per_sample = total_time / len(test_input_prompts)
    
    # Zip together outputs and inputs
    # pipeline_outputs is a list of dictionaries in the form [{"generated_text": "..."}, ...]
    batched_outputs_list = list(test_dataset)
    for i in range(len(test_input_prompts)):
        batched_outputs_list[i]["generated_text"] = pipeline_outputs[i][0]["generated_text"]

    # Make a check that the output is the same length as the test dataset
    assert len(batched_outputs_list) == len(test_dataset), f"Lengths of batched_output_list (len={len(batched_outputs_list)}) and test_dataset (len={len(test_dataset)}) don't line up"

    # Output statement of how we're going
    print(f"[INFO] Total inference time for batch size {BATCH_SIZE}: {total_time:.2f} seconds")
    print(f"[INFO] Average inference time per sample for batch size {BATCH_SIZE}: {avg_time_per_sample:.2f} seconds")
    print("="*80 + "\n\n")

    # Append results to the dictionary
    all_outputs_pipeline[BATCH_SIZE] = {"batched_outputs_list": batched_outputs_list, 
                                        "total_time": total_time,
                                        "avg_time_per_sample": avg_time_per_sample}


[INFO] Total number of samples to run through the pipeline: 284
[INFO] Running inference with batch size: 1
[INFO] Total pipeline outputs: 284
[INFO] Total inference time for batch size 1: 142.47 seconds
[INFO] Average inference time per sample for batch size 1: 0.50 seconds


[INFO] Running inference with batch size: 4
[INFO] Total pipeline outputs: 284
[INFO] Total inference time for batch size 4: 52.90 seconds
[INFO] Average inference time per sample for batch size 4: 0.19 seconds


[INFO] Running inference with batch size: 8
[INFO] Total pipeline outputs: 284
[INFO] Total inference time for batch size 8: 32.46 seconds
[INFO] Average inference time per sample for batch size 8: 0.11 seconds


[INFO] Running inference with batch size: 16
[INFO] Total pipeline outputs: 284
[INFO] Total inference time for batch size 16: 25.75 seconds
[INFO] Average inference time per sample for batch size 16: 0.09 seconds


[INFO] Running inference with batch size: 32
[INFO] Total pipeline outputs: 284


In [58]:
all_outputs_pipeline[32].keys()

dict_keys(['batched_outputs_list', 'total_time', 'avg_time_per_sample'])

In [59]:
for key, value in all_outputs_pipeline.items():
    print(f"Batch size: {key} | Total inference time: {value['total_time']:.2f} seconds | Total samples: {len(value['batched_outputs_list'])} | Average time per sample: {value['avg_time_per_sample']:.2f} seconds")

Batch size: 1 | Total inference time: 140.16 seconds | Total samples: 284 | Average time per sample: 0.49 seconds
Batch size: 4 | Total inference time: 53.58 seconds | Total samples: 284 | Average time per sample: 0.19 seconds
Batch size: 8 | Total inference time: 35.53 seconds | Total samples: 284 | Average time per sample: 0.13 seconds
Batch size: 16 | Total inference time: 28.67 seconds | Total samples: 284 | Average time per sample: 0.10 seconds
Batch size: 32 | Total inference time: 29.28 seconds | Total samples: 284 | Average time per sample: 0.10 seconds
Batch size: 64 | Total inference time: 31.48 seconds | Total samples: 284 | Average time per sample: 0.11 seconds
Batch size: 128 | Total inference time: 49.34 seconds | Total samples: 284 | Average time per sample: 0.17 seconds


## TK - Run batched inference with a KeyDataset

KeyDataset offers us the benefits of PyTorch's DataLoader behind the scenes and for working with larger datasets.

We don't have to turn our whole dataset into a list to iterate over it. 

We can lazily load batches of samples when we need to.

UPTOHERE:

* Explore keydataset docs
* Write code to handle the keydataset inference steps
* Later: eval the outputs based on sequence matching 
* Later: graphically assess the speed of the batched version versus the original version

In [61]:
test_dataset[0]

{'sequence': 'Living Planet Goat Milk Whole Milk, 1 Litre, GMO Free, Australian Dairy, 8.75g Protein Per Serve, Good Source of Calcium.',
 'image_url': None,
 'class_label': 'food',
 'source': 'manual_taken_images',
 'char_len': 121.0,
 'word_count': 20.0,
 'syn_or_real': 'syn',
 'uuid': '27d52571-269f-413a-97b5-fb1af9b34adc',
 'gpt-oss-120b-label': "{'is_food_or_drink': True, 'tags': ['np', 'fi', 'di', 'fp', 'fa'], 'food_items': ['Living Planet Goat Milk Whole Milk'], 'drink_items': ['Living Planet Goat Milk Whole Milk']}",
 'gpt-oss-120b-label-condensed': 'food_or_drink: 1\ntags: np, fi, di, fp, fa\nfoods: Living Planet Goat Milk Whole Milk\ndrinks: Living Planet Goat Milk Whole Milk',
 'target_food_names_to_use': None,
 'caption_detail_level': None,
 'num_foods': None,
 'target_image_point_of_view': None,
 'prompt': [{'content': 'Living Planet Goat Milk Whole Milk, 1 Litre, GMO Free, Australian Dairy, 8.75g Protein Per Serve, Good Source of Calcium.',
   'role': 'user'}],
 'completi

In [ ]:
import time
from transformers.pipelines.pt_utils import KeyDataset

from tqdm.auto import tqdm

all_outputs_keydataset = {}

BATCH_SIZES_TO_TEST = [1, 4, 8, 16, 32, 64, 128] # customize these based on the size of your model and GPU memory
# BATCH_SIZES_TO_TEST = [32]
VERBOSE = False

for BATCH_SIZE in BATCH_SIZES_TO_TEST:
    print(f"[INFO] Running KeyDataset batched inference with batch size: {BATCH_SIZE}")
    start_time = time.time()

    keydataset_outputs = []
    for i, keydataset_output in enumerate(tqdm(
        loaded_model_pipeline(KeyDataset(test_dataset, "formatted_prompt"),
                              batch_size=BATCH_SIZE,
                              max_new_tokens=256,
                              return_full_text=False),
        total=len(test_dataset),
        desc=f"Batch size: {BATCH_SIZE}"
    )):
        test_sample = dict(test_dataset[i])
        test_sample["generated_text"] = keydataset_output[0]["generated_text"]
        keydataset_outputs.append(test_sample)

    print(f"[INFO] Total KeyDataset outputs: {len(keydataset_outputs)}")

    end_time = time.time()
    total_time = end_time - start_time
    avg_time_per_sample = total_time / len(test_dataset)

    # Make a check that the output is the same length as the test dataset
    assert len(keydataset_outputs) == len(test_dataset), f"Lengths of batched_output_list (len={len(batched_outputs_list)}) and test_dataset (len={len(test_dataset)}) don't line up"

    # Output statement of how we're going
    print(f"[INFO] Total inference time for batch size {BATCH_SIZE}: {total_time:.2f} seconds")
    print(f"[INFO] Average inference time per sample for batch size {BATCH_SIZE}: {avg_time_per_sample:.2f} seconds")
    print("="*80 + "\n\n")

    # Save artifacts to dictionary
    all_outputs_keydataset[BATCH_SIZE] = {"batched_outputs_list": keydataset_outputs, 
                                          "total_time": total_time,
                                          "avg_time_per_sample": avg_time_per_sample}
    

[INFO] Running KeyDataset batched inference with batch size: 1


Batch size: 1:   0%|          | 0/284 [00:00<?, ?it/s]

[INFO] Total KeyDataset outputs: 284
[INFO] Total inference time for batch size 1: 142.65 seconds
[INFO] Average inference time per sample for batch size 1: 0.50 seconds


[INFO] Running KeyDataset batched inference with batch size: 4


Batch size: 4:   0%|          | 0/284 [00:00<?, ?it/s]

[INFO] Total KeyDataset outputs: 284
[INFO] Total inference time for batch size 4: 55.46 seconds
[INFO] Average inference time per sample for batch size 4: 0.20 seconds


[INFO] Running KeyDataset batched inference with batch size: 8


...

[INFO] Total KeyDataset outputs: 284
[INFO] Total inference time for batch size 32: 33.88 seconds
[INFO] Average inference time per sample for batch size 32: 0.12 seconds


[INFO] Running KeyDataset batched inference with batch size: 64


Batch size: 64:   0%|          | 0/284 [00:00<?, ?it/s]

[INFO] Total KeyDataset outputs: 284
[INFO] Total inference time for batch size 64: 31.20 seconds
[INFO] Average inference time per sample for batch size 64: 0.11 seconds


[INFO] Running KeyDataset batched inference with batch size: 128


Batch size: 128:   0%|          | 0/284 [00:00<?, ?it/s]

[INFO] Total KeyDataset outputs: 284
[INFO] Total inference time for batch size 128: 39.21 seconds
[INFO] Average inference time per sample for batch size 128: 0.14 seconds




TK - extension/question/callout: how might this process work for a sample of 1M items? perhaps we'd like to implement checkpointing as well as sample saving (e.g. every 50k samples) so our list doesn't get too big

In [69]:
print(f"[INFO] Showing information for KeyDataset batched input inference:")
for key, value in all_outputs_keydataset.items():
    print(f"Batch size: {key} | Total inference time: {value['total_time']:.2f} seconds | Total samples: {len(value['batched_outputs_list'])} | Average time per sample: {value['avg_time_per_sample']:.2f} seconds")

[INFO] Showing information for KeyDataset batched input inference:
Batch size: 1 | Total inference time: 142.65 seconds | Total samples: 284 | Average time per sample: 0.50 seconds
Batch size: 4 | Total inference time: 55.46 seconds | Total samples: 284 | Average time per sample: 0.20 seconds
Batch size: 8 | Total inference time: 33.07 seconds | Total samples: 284 | Average time per sample: 0.12 seconds
Batch size: 16 | Total inference time: 24.18 seconds | Total samples: 284 | Average time per sample: 0.09 seconds
Batch size: 32 | Total inference time: 33.88 seconds | Total samples: 284 | Average time per sample: 0.12 seconds
Batch size: 64 | Total inference time: 31.20 seconds | Total samples: 284 | Average time per sample: 0.11 seconds
Batch size: 128 | Total inference time: 39.21 seconds | Total samples: 284 | Average time per sample: 0.14 seconds


In [ ]:
# UPTOHERE:
# Next: 
# Build in evals for sequence matching - how close are the models to their original 1 sample batch?
    # Build in evals for comparing the generated outputs to the original outputs - how close are we? 
# Build in visual evals for speed

## TK - Eval with sequence matcher

In [120]:
from difflib import SequenceMatcher

def get_text_similarity(text_a, text_b, verbose=False):
    """Simple text-level overlap: proportion of matching texts/tokens."""
    split_a = text_a.split()
    split_b = text_b.split()
    if verbose:
        print(f"Text a: {split_a}")
        print(f"Text b: {split_b}")
    matcher = SequenceMatcher(None, split_a, split_b)
    return matcher.ratio() # 0.0 to 1.0

text_a_example = "Hello my name is Daniel!"
text_b_example = "Hello my name is Sam!"
text_c_example = "Hello my name is Daniel."
text_d_example = "Hi my name is daniel!"
text_e_example = "dfhwrh208r32048y3208yh"

print(get_text_similarity(text_a=text_a_example, text_b=text_a_example))
print(get_text_similarity(text_a=text_a_example, text_b=text_b_example))
print(get_text_similarity(text_a=text_a_example, text_b=text_c_example))
print(get_text_similarity(text_a=text_a_example, text_b=text_d_example))
print(get_text_similarity(text_a=text_a_example, text_b=text_e_example))

1.0
0.8
0.8
0.6
0.0


TK - note: you can create more creative evals here, we are just doing simple matching

In [102]:
all_outputs_keydataset.keys()

dict_keys([1, 4, 8, 16, 32, 64, 128])

In [103]:
baseline_results = all_outputs_keydataset[1]
baseline_total_time = baseline_results["total_time"]
baseline_avg_time_per_sample = baseline_results["avg_time_per_sample"]

def filter_generated_text(sample):
    prompt, generated_text = sample["formatted_prompt"], sample["generated_text"]
    if prompt in generated_text:
        return generated_text.replace(prompt, "")
    else:
        return generated_text

baseline_generated_texts = [filter_generated_text(sample) for sample in baseline_results["batched_outputs_list"]]
baseline_ground_truths = [sample["gpt-oss-120b-label-condensed"] for sample in baseline_results["batched_outputs_list"]]

In [104]:
baseline_sample = baseline_results["batched_outputs_list"][42]
baseline_sample

{'sequence': "This image shows the back of a food package with detailed cooking instructions, ingredients, and nutrition information. The package is held in a person's hand, and the background includes a concrete floor and part of a shoe.\n\n**Cooking Instructions:**\n- Ingredients listed include 1 cup snow peas, 1/2 cup frozen edamame, 2 cloves garlic, 2cm piece ginger, 1/4 cup stock of choice, 180g udon noodles, 3 spring onions, and 2 tablespoons sesame seeds.\n- Instructions mention cooking snow peas and edamame, adding oil, garlic, ginger, and stock, and then adding cooked udon noodles and tossing with protein, sesame seeds, and spring onions.\n\n**Nutrition Information:**\n- Servings per pack: 4\n- Serving size: 44g\n- Energy: 88kJ (21kcal) per serve, 201kJ (48kcal) per 100g\n- Protein: 0.4g per serve, 0.9g per 100g\n- Fat, Total: 0.6g per serve, 1.4g per 100g\n- Saturated Fat: 0.1g per serve, 0.3g per 100g\n- Carbohydrate: 3.4g per serve, 7.8g per 100g\n- Sugars: 1.1g per serve, 

In [105]:
filter_generated_text(baseline_sample)

'food_or_drink: 1\ntags: np, il, fi, fp\nfoods: snow peas, frozen edamame, garlic, ginger, stock of choice, udon noodles, spring onions, sesame seeds, oil, Garlic, Ginger, Xanthan gum, Natural Sweetener (Steviol glycosides), Spice Extract, Sesame, Soy\ndrinks: Water, Soy Sauce'

In [106]:
text_similarity(text_a=baseline_sample["gpt-oss-120b-label-condensed"],
                text_b=filter_generated_text(baseline_sample))

0.6732673267326733

In [ ]:
similarities = [
    get_text_similarity(text_a=ground_truth,
                       text_b=generated_text)
    for ground_truth, generated_text in zip(baseline_ground_truths, baseline_generated_texts)
]

avg_similarity_score = sum(similarities) / len(similarities) * 100
print(f"[INFO] Average similarity score for batch size 1 with ground truth: {avg_similarity_score:.2f}%")

[INFO] Average similarity score for batch size 1 with ground truth: 93.13%


In [ ]:
print(f"[INFO] Baseline total time for batch size 1: {baseline_total_time:.2f}s")

[INFO] Baseline total time for batch size 1: 142.65


In [114]:
all_outputs_keydataset[1].keys()

dict_keys(['batched_outputs_list', 'total_time', 'avg_time_per_sample'])

In [ ]:
# UPTOHERE:
# Calculating similarities between baseline + batching as well as batching -> ground truth
# Make the output of the results pretty
# Rerun to make sure everything works
# Later: Make the speedups visual with a graph

In [130]:
for BATCH_SIZE in BATCH_SIZES_TO_TEST:
    target_outputs = all_outputs_keydataset[BATCH_SIZE]
    target_generated_texts = [filter_generated_text(sample) for sample in target_outputs["batched_outputs_list"]]
    
    # Do each of the batch sizes have the same number of samples?
    len_match = len(baseline_generated_texts) == len(target_generated_texts)

    # Measure the similarities across batches
    similarity_scores = [
        get_text_similarity(text_a=baseline_text, text_b=target_text)
        for baseline_text, target_text in zip(baseline_generated_texts, target_generated_texts)
    ]
    avg_similarity_score = round(sum(similarity_scores) / len(similarity_scores) * 100, 2) 

    speedup = round(baseline_total_time / target_outputs["total_time"], 2)


    print(f"[INFO] Comparing batch size 1 to {BATCH_SIZE}:")
    print(f"Do lengths match? {len_match}")
    print(f"Avg similarity score: {avg_similarity_score}%")
    print(f"Speedup: {speedup}x")
    print()

[INFO] Comparing batch size 1 to 1:
Do lengths match? True
Avg similarity score: 100.0%
Speedup: 1.0x

[INFO] Comparing batch size 1 to 4:
Do lengths match? True
Avg similarity score: 95.88%
Speedup: 2.57x

[INFO] Comparing batch size 1 to 8:
Do lengths match? True
Avg similarity score: 95.73%
Speedup: 4.31x

[INFO] Comparing batch size 1 to 16:
Do lengths match? True
Avg similarity score: 96.28%
Speedup: 5.9x

[INFO] Comparing batch size 1 to 32:
Do lengths match? True
Avg similarity score: 95.37%
Speedup: 4.21x

[INFO] Comparing batch size 1 to 64:
Do lengths match? True
Avg similarity score: 95.58%
Speedup: 4.57x

[INFO] Comparing batch size 1 to 128:
Do lengths match? True
Avg similarity score: 96.22%
Speedup: 3.64x



In [124]:
target_generated_texts[:10]

['food_or_drink: 1\ntags: np, fi, fa\nfoods: Goat Milk Whole Milk, GMO Free, Australian Dairy, 8.75g Protein Per Serve\ndrinks:',
 "food_or_drink: 1\ntags: np, il, fi, fp\nfoods: Fisherman's Friend Original Extra Strong Menthol Lozenges, Sugar, Flavourings: Liquorice Extract, Eucalyptus Oil, Capsicum Tincture, Dextrin, Gum Tragacanth\ndrinks:",
 'food_or_drink: 1\ntags: np, il, fi, fp\nfoods: Flour (Wheat), Butter (Cream (Milk), Milk Solids, Cultures), Salt, Neapolitan, Choc Vanilla, ice cream products, Neapolitan, Choc Vanilla\ndrinks:',
 'food_or_drink: 1\ntags: np, il, fi, fp\nfoods: Potatoes, Olive Oil, Sea Salt\ndrinks:',
 "food_or_drink: 1\ntags: il, fi, fa\nfoods: Wheat flour, water, vegetable oil, peri peri chilli puree (3%) [bird's eye chilli, red cayenne chilli], humectant (422), cultured wheat flour, vegetable emulsifiers (471, 481), wheat gluten, raising agents (341, 450, 500), iodised salt, acidity regulator (297), chilli powder, soy flour, stabiliser (412), vitamins (thia

In [125]:
baseline_generated_texts[:10]

['food_or_drink: 1\ntags: fi\nfoods: Goat Milk, Milk\ndrinks:',
 "food_or_drink: 1\ntags: np, il, fi, fp\nfoods: Fisherman's Friend Original Extra Strong Menthol Lozenges, Sugar, Flavourings: Liquorice Extract, Eucalyptus Oil, Capsicum Tincture, Dextrin, Gum Tragacanth, Menthol, Thickeners: Eucalyptus Oil, Capsicum Tincture, Dextrin, Gum Tragacanth\ndrinks:",
 'food_or_drink: 1\ntags: np, il, fi, fp\nfoods: Flour (Wheat), Butter (Cream (Milk), Milk Solids, Cultures), Salt, Neapolitan, Choc Vanilla flavors, Milk, Cream (Milk), Milk Solids, Cultures, Salt\ndrinks:',
 'food_or_drink: 1\ntags: np, il, fi, fp\nfoods: Potatoes, Olive Oil, Sea Salt\ndrinks:',
 'food_or_drink: 1\ntags: il, fi, fp\nfoods: Wheat flour, water, vegetable oil, peri peri chilli puree, chilli chilli, red cayenne chilli, humectant (422), cultured wheat flour, vegetable emulsifiers (471, 481), wheat gluten, raising agents (341, 450, 500), iodised salt, acidity regulator (297), chilli powder, soy flour, stabiliser (412)

In [117]:
for sample in target_outputs["batched_outputs_list"]:
    print(sample)

{'sequence': 'Living Planet Goat Milk Whole Milk, 1 Litre, GMO Free, Australian Dairy, 8.75g Protein Per Serve, Good Source of Calcium.', 'image_url': None, 'class_label': 'food', 'source': 'manual_taken_images', 'char_len': 121.0, 'word_count': 20.0, 'syn_or_real': 'syn', 'uuid': '27d52571-269f-413a-97b5-fb1af9b34adc', 'gpt-oss-120b-label': "{'is_food_or_drink': True, 'tags': ['np', 'fi', 'di', 'fp', 'fa'], 'food_items': ['Living Planet Goat Milk Whole Milk'], 'drink_items': ['Living Planet Goat Milk Whole Milk']}", 'gpt-oss-120b-label-condensed': 'food_or_drink: 1\ntags: np, fi, di, fp, fa\nfoods: Living Planet Goat Milk Whole Milk\ndrinks: Living Planet Goat Milk Whole Milk', 'target_food_names_to_use': None, 'caption_detail_level': None, 'num_foods': None, 'target_image_point_of_view': None, 'prompt': [{'content': 'Living Planet Goat Milk Whole Milk, 1 Litre, GMO Free, Australian Dairy, 8.75g Protein Per Serve, Good Source of Calcium.', 'role': 'user'}], 'completion': [{'content': 

In [ ]:
# Get batch size 1 as default outputs
# Comparisons: 
    # Compare to batch size 1
    # Compare to original label (simple matching)
    # Compare time and speedup multiplier

In [ ]:
# Batch Size   Length Match   Avg Token Sim    Time (s)   Speedup vs bs=1
# ======================================================================
# 1            True           100.00%          138.49     1.00x
# 4            True           99.08%           52.81      2.62x
# 8            True           99.20%           32.97      4.20x
# 16           True           99.17%           25.95      5.34x
# 32           True           99.13%           23.01      6.02x
# 64           True           99.25%           28.02      4.94x
# 128          True           99.24%           40.81      3.39x

## TK - Eval speed with visual processing